# Semi-Blind PSF Deconvolution for Microscopy

## Problem Background and Motivation

In optical microscopy, the observed image is fundamentally degraded by the **Point Spread Function (PSF)** of the imaging system. The PSF describes how a point source of light is spread out in the recorded image due to diffraction, aberrations, and other optical imperfections. This blurring effect limits the resolution and contrast of microscopy images, making it difficult to resolve fine cellular structures.

### The Challenge of Spatially Variant Blur

In real microscopy systems, the PSF is often **spatially variant** — it changes across the field of view due to:
- **Optical aberrations** that vary with position
- **Sample-induced aberrations** from refractive index variations
- **Depth-dependent blur** in thick specimens
- **Defocus variations** across tilted samples

This makes deconvolution significantly more challenging than the spatially invariant case, where a single PSF applies uniformly across the image.

### Why This is an Inverse Problem

The forward imaging process (blurring) is well-defined, but recovering the original sharp image from the blurred observation is an **ill-posed inverse problem**:
1. **Non-uniqueness**: Multiple sharp images could produce similar blurred observations
2. **Instability**: Small noise in the observation can cause large errors in reconstruction
3. **Incomplete information**: High-frequency details are attenuated by the PSF

### Semi-Blind Deconvolution

In **semi-blind deconvolution**, we don't know the exact PSF but have some prior knowledge about its structure. Our approach uses:
1. A **deep neural network** to estimate local PSF parameters from image patches
2. **Alternating optimization** between PSF estimation and image reconstruction
3. **Richardson-Lucy deconvolution** with Total Variation regularization for the reconstruction step

### Historical Context and References

The Richardson-Lucy algorithm was independently developed by Richardson (1972) and Lucy (1974) for astronomical image restoration. The extension to spatially variant PSFs and the incorporation of deep learning for PSF estimation represents modern advances in computational microscopy.

**Key References:**
- Richardson, W.H. (1972). "Bayesian-based iterative method of image restoration." JOSA 62(1):55-59
- Lucy, L.B. (1974). "An iterative technique for the rectification of observed distributions." AJ 79:745
- Lauer, T. (2002). "Deconvolution with a spatially-variant PSF." SPIE 4847

## Mathematical Formulation

### Forward Model: Spatially Variant Convolution

For a spatially variant imaging system, the observed image $y$ is related to the true image $x$ by:

$$y(\mathbf{r}) = \int h(\mathbf{r}, \mathbf{r}') x(\mathbf{r}') d\mathbf{r}' + n(\mathbf{r}) \tag{1}$$

where $h(\mathbf{r}, \mathbf{r}')$ is the spatially variant PSF that depends on both the observation position $\mathbf{r}$ and the source position $\mathbf{r}'$, and $n$ is additive noise.

### Discrete Approximation with PSF Grid

We approximate the spatially variant PSF using a grid of $K$ local PSFs $\{h_k\}_{k=1}^K$ with interpolation weights $\{w_k(\mathbf{r})\}$:

$$y(\mathbf{r}) \approx \sum_{k=1}^{K} w_k(\mathbf{r}) \cdot (h_k * x)(\mathbf{r}) + n(\mathbf{r}) \tag{2}$$

where $*$ denotes standard convolution and the weights satisfy $\sum_k w_k(\mathbf{r}) = 1$ for all positions.

### Gaussian PSF Parameterization

Each local PSF is modeled as an anisotropic Gaussian characterized by its Full Width at Half Maximum (FWHM) in $x$ and $y$ directions:

$$h_k(u, v) = \frac{1}{Z_k} \exp\left(-4\ln(2)\left[\frac{(u-u_0)^2}{\sigma_{x,k}^2} + \frac{(v-v_0)^2}{\sigma_{y,k}^2}\right]\right) \tag{3}$$

where $\sigma_{x,k}$ and $\sigma_{y,k}$ are the FWHM parameters and $Z_k$ is a normalization constant ensuring $\sum h_k = 1$.

### Inverse Problem: Maximum A Posteriori Estimation

We seek to recover $x$ by solving the optimization problem:

$$\hat{x} = \arg\min_{x \geq 0} \left\{ -\sum_{\mathbf{r}} \left[ y(\mathbf{r}) \log(\hat{y}(\mathbf{r})) - \hat{y}(\mathbf{r}) \right] + \lambda \cdot \text{TV}(x) \right\} \tag{4}$$

where $\hat{y} = \sum_k w_k \cdot (h_k * x)$ is the predicted observation, the first term is the Poisson log-likelihood (appropriate for photon-counting detectors), and $\text{TV}(x) = \|\nabla x\|_1$ is the Total Variation regularizer.

### Richardson-Lucy Update with TV Regularization

The iterative update rule combines the Richardson-Lucy algorithm with TV regularization:

$$x^{(n+1)} = \frac{x^{(n)}}{1 - \lambda \cdot \text{div}\left(\frac{\nabla x^{(n)}}{\|\nabla x^{(n)}\|}\right)} \cdot \sum_k w_k \cdot \left( h_k^T * \frac{y}{\sum_j w_j \cdot (h_j * x^{(n)})} \right) \tag{5}$$

where $h_k^T$ denotes the flipped (adjoint) PSF and $\text{div}$ is the divergence operator.

### PSF Parameter Estimation via Deep Learning

The local PSF parameters $(\sigma_x, \sigma_y)$ are estimated from image patches using a ResNet-based classifier:

$$\boldsymbol{\theta}_k = f_{\text{CNN}}(\text{patch}_k; \mathbf{W}) \tag{6}$$

where $f_{\text{CNN}}$ is the trained neural network with weights $\mathbf{W}$, and $\boldsymbol{\theta}_k = (\sigma_{x,k}, \sigma_{y,k})$ are the estimated PSF parameters for patch $k$.

In [ ]:
# ============================================
# Cell 3: Environment Setup & Imports
# ============================================

import numpy as np
import scipy.fftpack
import scipy.ndimage
from numpy.fft import rfft2, irfft2
from scipy.interpolate import griddata
from functools import reduce
import matplotlib.pyplot as plt
from skimage import metrics
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)

# Configure matplotlib for publication-quality figures
plt.rcParams.update({
    'figure.figsize': (12, 8),
    'figure.dpi': 100,
    'font.size': 12,
    'font.family': 'serif',
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'image.cmap': 'gray',
    'image.interpolation': 'nearest'
})

# Print library versions
print("Library Versions:")
print(f"  NumPy: {np.__version__}")
print(f"  SciPy: {scipy.__version__}")
import skimage
print(f"  scikit-image: {skimage.__version__}")
import matplotlib
print(f"  Matplotlib: {matplotlib.__version__}")
print("\nEnvironment setup complete!")

## Forward Model Explanation

### Physics of Spatially Variant Blur in Microscopy

The forward model simulates how a microscope images a specimen with spatially varying optical properties. The key physical processes are:

#### 1. Point Spread Function (PSF)

The PSF describes the response of the imaging system to a point source. For a diffraction-limited system, the PSF is approximately an Airy pattern, which we approximate with a Gaussian:

$$h(u, v; \sigma_x, \sigma_y) = \frac{1}{2\pi\sigma_x\sigma_y} \exp\left(-\frac{u^2}{2\sigma_x^2} - \frac{v^2}{2\sigma_y^2}\right)$$

The FWHM (Full Width at Half Maximum) relates to $\sigma$ by: $\text{FWHM} = 2\sqrt{2\ln 2} \cdot \sigma \approx 2.355\sigma$

#### 2. Spatial Variation

In real systems, the PSF varies across the field of view due to:
- **Field curvature**: Focus varies with distance from optical axis
- **Astigmatism**: Different focal lengths in orthogonal directions
- **Coma and spherical aberration**: Asymmetric blur patterns

We model this by defining PSFs at grid points and interpolating between them.

#### 3. Interpolation Scheme

For a grid of $M \times N$ PSF samples, we compute bilinear interpolation weights $w_k(\mathbf{r})$ such that:

$$\sum_{k=1}^{MN} w_k(\mathbf{r}) = 1 \quad \forall \mathbf{r}$$

The blurred image is then:

$$y = \sum_{k=1}^{MN} w_k \odot (h_k * x) + n$$

where $\odot$ denotes element-wise multiplication.

#### 4. Noise Model

We add Gaussian noise to simulate detector readout noise:

$$n \sim \mathcal{N}(0, \sigma_n^2)$$

In practice, microscopy images also exhibit Poisson noise from photon statistics, but Gaussian noise is a reasonable approximation for high photon counts.

In [ ]:
# ============================================
# Cell 5: Forward Model & Synthetic Data Generation
# ============================================

def normalize(v):
    """Normalize a 2D matrix to sum to 1."""
    norm = v.sum()
    if norm == 0:
        norm = np.finfo(v.dtype).eps
    return v / norm

def gaussian_kernel(size, fwhmx=3, fwhmy=3, center=None):
    """Generate a normalized 2D Gaussian kernel."""
    x = np.arange(0, size, 1, float)
    y = x[:, np.newaxis]
    
    if center is None:
        x0 = y0 = size // 2
    else:
        x0, y0 = center
    
    kernel = np.exp(-4 * np.log(2) * (((x - x0) ** 2) / fwhmx**2 + 
                                       ((y - y0) ** 2) / fwhmy**2))
    return normalize(kernel)

def compute_grid_weights(psf_map_shape, image_shape):
    """Compute bilinear interpolation weights for PSF grid."""
    grid_weights = []
    grid_x, grid_y = np.mgrid[0:image_shape[0], 0:image_shape[1]]
    xmax = np.linspace(0, image_shape[0], psf_map_shape[0])
    ymax = np.linspace(0, image_shape[1], psf_map_shape[1])
    
    total_patches = psf_map_shape[0] * psf_map_shape[1]
    
    for i in range(total_patches):
        points = []
        values = []
        for x in xmax:
            for y in ymax:
                points.append(np.asarray([x, y]))
                values.append(0.0)
        values[i] = 1.0
        points = np.asarray(points)
        values = np.asarray(values)
        weight = griddata(points, values, (grid_x, grid_y), 
                         method='linear', rescale=True)
        grid_weights.append(weight)
    
    return grid_weights

def _centered(arr, newshape):
    """Extract centered region of array."""
    newshape = np.asarray(newshape)
    currshape = np.array(arr.shape)
    startind = (currshape - newshape) // 2
    endind = startind + newshape
    myslice = [slice(startind[k], endind[k]) for k in range(len(endind))]
    return arr[tuple(myslice)]

def unpad(img, npad):
    """Remove padding from image."""
    return img[npad:-npad, npad:-npad]

def apply_spatially_variant_blur(image, psf_list, grid_weights):
    """Apply spatially variant blur using PSF grid and interpolation weights."""
    blurred_image = np.zeros_like(image)
    
    # Pad image for convolution
    pad_width = np.max(psf_list[0].shape)
    img_padded = np.pad(image, pad_width, mode='reflect')
    
    # Compute FFT sizes
    fft_shape = np.array(img_padded.shape) + np.array(psf_list[0].shape) - 1
    fsize = [scipy.fftpack.next_fast_len(int(d)) for d in fft_shape]
    fslice = tuple([slice(0, int(sz)) for sz in fft_shape])
    
    img_f = rfft2(img_padded, fsize)
    
    for i, psf in enumerate(psf_list):
        psf_f = rfft2(psf, fsize)
        convolved = irfft2(np.multiply(psf_f, img_f), fsize)[fslice].real
        convolved = _centered(convolved, img_padded.shape)
        convolved = unpad(convolved, pad_width)
        blurred_image += convolved * grid_weights[i]
    
    return blurred_image

# Generate synthetic ground truth image
print("Generating synthetic microscopy image...")
image_size = 256

# Create a synthetic "cell" image with various structures
gt_img = np.zeros((image_size, image_size))

# Add circular structures (simulating cell nuclei)
xx, yy = np.mgrid[0:image_size, 0:image_size]
centers = [(64, 64), (192, 64), (64, 192), (192, 192), (128, 128)]
radii = [20, 25, 22, 18, 30]
intensities = [0.8, 0.9, 0.7, 0.85, 1.0]

for (cx, cy), r, intensity in zip(centers, radii, intensities):
    mask = ((xx - cx)**2 + (yy - cy)**2) < r**2
    gt_img[mask] = intensity

# Add some fine structures (simulating filaments)
for i in range(5):
    x_start = np.random.randint(30, image_size-30)
    y_start = np.random.randint(30, image_size-30)
    length = np.random.randint(30, 60)
    angle = np.random.uniform(0, np.pi)
    for t in np.linspace(0, length, 50):
        x = int(x_start + t * np.cos(angle))
        y = int(y_start + t * np.sin(angle))
        if 0 <= x < image_size and 0 <= y < image_size:
            gt_img[x, y] = 0.6

# Add point sources (simulating fluorescent markers)
for _ in range(20):
    x, y = np.random.randint(10, image_size-10, 2)
    gt_img[x-1:x+2, y-1:y+2] = 0.95

# Smooth slightly to make it more realistic
gt_img = scipy.ndimage.gaussian_filter(gt_img, sigma=1.0)
gt_img = np.clip(gt_img, 0, 1)

print(f"Ground truth image shape: {gt_img.shape}")

# Define spatially variant PSF grid (2x2)
print("\nCreating spatially variant PSF grid...")
grid_rows, grid_cols = 2, 2
psf_size = 31

# FWHM varies across the field of view
# Top-left: sharp, Bottom-right: more blurred
fwhm_grid = [
    (2.0, 2.0),   # Top-left: nearly symmetric, small blur
    (2.5, 4.0),   # Top-right: astigmatic
    (3.5, 2.5),   # Bottom-left: astigmatic (other direction)
    (4.5, 4.5),   # Bottom-right: large symmetric blur
]

psf_list = []
print("PSF parameters at grid points:")
for idx, (fx, fy) in enumerate(fwhm_grid):
    psf = gaussian_kernel(psf_size, fx, fy)
    psf_list.append(psf)
    row, col = idx // grid_cols, idx % grid_cols
    print(f"  Grid ({row}, {col}): FWHM_x = {fx:.1f}, FWHM_y = {fy:.1f}")

# Compute interpolation weights
print("\nComputing interpolation weights...")
grid_weights = compute_grid_weights((grid_rows, grid_cols), gt_img.shape)

# Apply spatially variant blur
print("Applying spatially variant blur...")
blurred_img = apply_spatially_variant_blur(gt_img, psf_list, grid_weights)

# Add noise
noise_sigma = 0.01
noisy_blurred_img = blurred_img + np.random.normal(0, noise_sigma, blurred_img.shape)
noisy_blurred_img = np.clip(noisy_blurred_img, 0, 1)

print(f"Noise level (sigma): {noise_sigma}")
print(f"Blurred image range: [{noisy_blurred_img.min():.4f}, {noisy_blurred_img.max():.4f}]")

# Visualize forward model
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Top row: PSFs
for idx, psf in enumerate(psf_list):
    ax = axes[0, idx]
    im = ax.imshow(psf, cmap='hot')
    ax.set_title(f'PSF {idx+1}\nFWHM: ({fwhm_grid[idx][0]:.1f}, {fwhm_grid[idx][1]:.1f})')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

# Bottom row: Images
ax = axes[1, 0]
im = ax.imshow(gt_img, cmap='gray')
ax.set_title('Ground Truth')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046)

ax = axes[1, 1]
im = ax.imshow(blurred_img, cmap='gray')
ax.set_title('Blurred (no noise)')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046)

ax = axes[1, 2]
im = ax.imshow(noisy_blurred_img, cmap='gray')
ax.set_title(f'Blurred + Noise (σ={noise_sigma})')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046)

# Show weight map for one PSF
ax = axes[1, 3]
im = ax.imshow(grid_weights[0], cmap='viridis')
ax.set_title('Weight Map (PSF 1)')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Forward Model: Spatially Variant Blur', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nForward model complete!")

## Reconstruction Algorithm

### Overview: Alternating Optimization

Our semi-blind deconvolution approach alternates between two steps:
1. **PSF Estimation**: Estimate local PSF parameters from image patches
2. **Image Reconstruction**: Given estimated PSFs, recover the sharp image

In this tutorial, we simulate the PSF estimation step (which would normally use a trained CNN) and focus on the reconstruction algorithm.

### Richardson-Lucy Algorithm

The Richardson-Lucy (RL) algorithm is derived from maximum likelihood estimation under a Poisson noise model. For spatially invariant blur:

$$x^{(n+1)} = x^{(n)} \cdot \left( h^T * \frac{y}{h * x^{(n)}} \right)$$

where $h^T$ is the transposed (flipped) PSF.

### Extension to Spatially Variant PSF

For spatially variant blur with PSF grid $\{h_k\}$ and weights $\{w_k\}$:

1. **Forward projection**: Compute predicted observation
$$\hat{y}^{(n)} = \sum_k w_k \odot (h_k * x^{(n)})$$

2. **Ratio computation**: 
$$r^{(n)} = \frac{y}{\hat{y}^{(n)}}$$

3. **Back projection** for each PSF region:
$$e_k^{(n)} = h_k^T * r^{(n)}$$

4. **Update with weighted combination**:
$$x^{(n+1)} = x^{(n)} \cdot \sum_k w_k \odot e_k^{(n)}$$

### Total Variation Regularization

To suppress noise amplification, we add TV regularization:

$$\text{TV}(x) = \sum_{i,j} \sqrt{(x_{i+1,j} - x_{i,j})^2 + (x_{i,j+1} - x_{i,j})^2}$$

The regularization term modifies the update:

$$x^{(n+1)} = \frac{x^{(n)} \cdot \text{(RL update)}}{1 - \lambda \cdot \text{div}\left(\frac{\nabla x^{(n)}}{|\nabla x^{(n)}|}\right)}$$

### Convergence Properties

- The RL algorithm is **guaranteed to converge** to a non-negative solution
- It preserves flux (total intensity)
- Without regularization, it tends to **amplify noise** at high iterations
- TV regularization provides **edge-preserving smoothing**

### Hyperparameter Choices

- **Number of iterations**: 10-30 typically sufficient; more iterations = sharper but noisier
- **TV weight $\lambda$**: 0.01-0.2; higher = smoother result
- **PSF grid resolution**: Trade-off between accuracy and computation

In [ ]:
# ============================================
# Cell 7: Reconstruction Implementation
# ============================================

def div0(a, b):
    """Safe division, returning 0 where division is undefined."""
    with np.errstate(divide='ignore', invalid='ignore'):
        c = np.true_divide(a, b)
        c[~np.isfinite(c)] = 0
    return c

def divergence(F):
    """Compute divergence of a 2D field."""
    return reduce(np.add, np.gradient(F))

def rl_deconv_spatially_variant(img_list, psf_list, iterations=10, lbd=0.1, 
                                 track_convergence=True):
    """
    Spatially-Variant Richardson-Lucy deconvolution with TV regularization.
    
    Parameters:
    -----------
    img_list : list of ndarray
        List of weighted observation images (w_k * y)
    psf_list : list of ndarray
        List of PSFs for each grid point
    iterations : int
        Number of RL iterations
    lbd : float
        TV regularization weight
    track_convergence : bool
        Whether to track convergence metrics
    
    Returns:
    --------
    result : ndarray
        Reconstructed image
    history : dict
        Convergence history (if track_convergence=True)
    """
    # Initialize
    min_values = []
    processed_imgs = []
    
    # Pad images
    pad_width = np.max(psf_list[0].shape)
    for img in img_list:
        img_padded = np.pad(img, pad_width, mode='reflect')
        min_val = np.min(img)
        img_padded = img_padded - min_val
        min_values.append(min_val)
        processed_imgs.append(img_padded)
    
    # FFT sizes
    fft_shape = np.array(processed_imgs[0].shape) + np.array(psf_list[0].shape) - 1
    fsize = [scipy.fftpack.next_fast_len(int(d)) for d in fft_shape]
    fslice = tuple([slice(0, int(sz)) for sz in fft_shape])
    
    # Initialize estimates
    latent_estimate = [img.copy() for img in processed_imgs]
    error_estimate = [img.copy() for img in processed_imgs]
    
    # Precompute PSF FFTs
    psf_f = []
    psf_flipped_f = []
    for psf in psf_list:
        psf_f.append(rfft2(psf, fsize))
        psf_flipped = np.flip(np.flip(psf, axis=0), axis=1)
        psf_flipped_f.append(rfft2(psf_flipped, fsize))
    
    # Convergence tracking
    history = {'iteration': [], 'residual': [], 'tv_norm': [], 'change': []}
    prev_estimate = None
    
    # Main iteration loop
    for it in range(iterations):
        regularization = np.ones(processed_imgs[0].shape)
        total_residual = 0
        
        for idx, img in enumerate(latent_estimate):
            # Forward projection: convolve estimate with PSF
            estimate_f = rfft2(latent_estimate[idx], fsize)
            estimate_conv = irfft2(np.multiply(psf_f[idx], estimate_f), fsize)[fslice].real
            estimate_conv = _centered(estimate_conv, img.shape)
            
            # Compute ratio (relative blur)
            relative_blur = div0(processed_imgs[idx], estimate_conv + 1e-10)
            total_residual += np.sum((processed_imgs[idx] - estimate_conv)**2)
            
            # Back projection
            relative_blur_f = rfft2(relative_blur, fsize)
            error_estimate[idx] = irfft2(np.multiply(psf_flipped_f[idx], relative_blur_f), 
                                         fsize)[fslice].real
            error_estimate[idx] = _centered(error_estimate[idx], img.shape)
            
            # TV regularization term
            norm_factor = np.linalg.norm(latent_estimate[idx], ord=1) + 1e-10
            div_val = divergence(latent_estimate[idx] / norm_factor)
            regularization += 1.0 - (lbd * div_val)
            
            # Multiplicative update
            latent_estimate[idx] = np.multiply(latent_estimate[idx], error_estimate[idx])
        
        # Apply regularization
        for idx in range(len(processed_imgs)):
            latent_estimate[idx] = np.divide(latent_estimate[idx], 
                                             regularization / float(len(processed_imgs)))
        
        # Track convergence
        if track_convergence:
            current_sum = np.sum(latent_estimate, axis=0)
            tv_norm = np.sum(np.abs(np.gradient(current_sum)[0])) + \
                      np.sum(np.abs(np.gradient(current_sum)[1]))
            
            if prev_estimate is not None:
                change = np.linalg.norm(current_sum - prev_estimate) / \
                        (np.linalg.norm(prev_estimate) + 1e-10)
            else:
                change = 1.0
            
            history['iteration'].append(it + 1)
            history['residual'].append(total_residual)
            history['tv_norm'].append(tv_norm)
            history['change'].append(change)
            
            prev_estimate = current_sum.copy()
            
            if (it + 1) % 5 == 0 or it == 0:
                print(f"  Iteration {it+1:3d}: Residual = {total_residual:.2e}, "
                      f"TV = {tv_norm:.2e}, Change = {change:.4f}")
    
    # Unpad and restore values
    for idx in range(len(latent_estimate)):
        latent_estimate[idx] += min_values[idx]
        latent_estimate[idx] = unpad(latent_estimate[idx], pad_width)
    
    result = np.sum(latent_estimate, axis=0)
    
    return result, history

# Simulate PSF estimation (in practice, this would use a trained CNN)
print("Simulating PSF estimation...")
print("(In the full pipeline, a ResNet would estimate these from image patches)")

# For this demo, we use slightly perturbed versions of the true PSF parameters
# to simulate estimation error
estimated_fwhm_grid = []
print("\nEstimated vs True PSF parameters:")
for idx, (true_fx, true_fy) in enumerate(fwhm_grid):
    # Add some estimation error
    est_fx = true_fx + np.random.uniform(-0.3, 0.3)
    est_fy = true_fy + np.random.uniform(-0.3, 0.3)
    est_fx = max(0.5, est_fx)  # Ensure positive
    est_fy = max(0.5, est_fy)
    estimated_fwhm_grid.append((est_fx, est_fy))
    print(f"  PSF {idx+1}: True ({true_fx:.2f}, {true_fy:.2f}) -> "
          f"Est ({est_fx:.2f}, {est_fy:.2f})")

# Create estimated PSFs
estimated_psf_list = []
for fx, fy in estimated_fwhm_grid:
    psf = gaussian_kernel(psf_size, fx, fy)
    estimated_psf_list.append(psf)

# Prepare weighted images for reconstruction
print("\nPreparing weighted images for reconstruction...")
image_masked_list = []
for i in range(len(estimated_psf_list)):
    image_masked_list.append(np.multiply(grid_weights[i], noisy_blurred_img))

# Run reconstruction
print("\nRunning Richardson-Lucy deconvolution with TV regularization...")
print(f"Parameters: iterations=20, lambda={0.05}")
print("="*60)

reconstructed_img, convergence_history = rl_deconv_spatially_variant(
    image_masked_list, 
    estimated_psf_list, 
    iterations=20, 
    lbd=0.05,
    track_convergence=True
)

# Clip to valid range
reconstructed_img = np.clip(reconstructed_img, 0, 1)

print("="*60)
print("Reconstruction complete!")
print(f"Output range: [{reconstructed_img.min():.4f}, {reconstructed_img.max():.4f}]")

## Results Visualization

In this section, we visualize the reconstruction results and compare them with the ground truth and blurred observation.

### What to Look For

1. **Sharpness Recovery**: The reconstructed image should show sharper edges and finer details compared to the blurred input

2. **Spatial Variation**: Different regions of the image had different blur levels. Check if the reconstruction quality is consistent across the image or if some regions are better recovered than others

3. **Noise Behavior**: The TV regularization should suppress noise while preserving edges. Look for any noise amplification or over-smoothing

4. **Artifact Detection**: Common artifacts include:
   - Ringing near sharp edges
   - Checkerboard patterns from FFT operations
   - Boundary effects from padding

### Quantitative Metrics

We compute:
- **PSNR (Peak Signal-to-Noise Ratio)**: Higher is better; measures overall reconstruction fidelity
- **SSIM (Structural Similarity Index)**: Higher is better; measures perceptual quality

In [ ]:
# ============================================
# Cell 9: Visualization - Ground Truth vs Reconstruction
# ============================================

# Compute metrics
psnr_blurred = metrics.peak_signal_noise_ratio(gt_img, noisy_blurred_img, data_range=1.0)
ssim_blurred = metrics.structural_similarity(gt_img, noisy_blurred_img, data_range=1.0)

psnr_reconstructed = metrics.peak_signal_noise_ratio(gt_img, reconstructed_img, data_range=1.0)
ssim_reconstructed = metrics.structural_similarity(gt_img, reconstructed_img, data_range=1.0)

print("Quantitative Metrics:")
print("="*50)
print(f"Blurred Image:      PSNR = {psnr_blurred:.2f} dB, SSIM = {ssim_blurred:.4f}")
print(f"Reconstructed:      PSNR = {psnr_reconstructed:.2f} dB, SSIM = {ssim_reconstructed:.4f}")
print(f"Improvement:        ΔPSNR = {psnr_reconstructed - psnr_blurred:+.2f} dB, "
      f"ΔSSIM = {ssim_reconstructed - ssim_blurred:+.4f}")
print("="*50)

# Create comparison figure
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Top row: Full images
ax = axes[0, 0]
im = ax.imshow(gt_img, cmap='gray', vmin=0, vmax=1)
ax.set_title('Ground Truth', fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[0, 1]
im = ax.imshow(noisy_blurred_img, cmap='gray', vmin=0, vmax=1)
ax.set_title(f'Blurred + Noise\nPSNR: {psnr_blurred:.2f} dB, SSIM: {ssim_blurred:.4f}', 
             fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[0, 2]
im = ax.imshow(reconstructed_img, cmap='gray', vmin=0, vmax=1)
ax.set_title(f'Reconstructed\nPSNR: {psnr_reconstructed:.2f} dB, SSIM: {ssim_reconstructed:.4f}', 
             fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Bottom row: Zoomed regions (center crop)
crop_size = 80
cx, cy = image_size // 2, image_size // 2
crop_slice = (slice(cx-crop_size//2, cx+crop_size//2), 
              slice(cy-crop_size//2, cy+crop_size//2))

ax = axes[1, 0]
im = ax.imshow(gt_img[crop_slice], cmap='gray', vmin=0, vmax=1)
ax.set_title('Ground Truth (Zoomed)', fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[1, 1]
im = ax.imshow(noisy_blurred_img[crop_slice], cmap='gray', vmin=0, vmax=1)
ax.set_title('Blurred (Zoomed)', fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[1, 2]
im = ax.imshow(reconstructed_img[crop_slice], cmap='gray', vmin=0, vmax=1)
ax.set_title('Reconstructed (Zoomed)', fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('Semi-Blind Deconvolution Results', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Line profile comparison
fig, ax = plt.subplots(figsize=(12, 4))
profile_row = image_size // 2
ax.plot(gt_img[profile_row, :], 'g-', linewidth=2, label='Ground Truth', alpha=0.8)
ax.plot(noisy_blurred_img[profile_row, :], 'r--', linewidth=1.5, label='Blurred', alpha=0.7)
ax.plot(reconstructed_img[profile_row, :], 'b-', linewidth=1.5, label='Reconstructed', alpha=0.8)
ax.set_xlabel('Pixel Position')
ax.set_ylabel('Intensity')
ax.set_title(f'Intensity Profile (Row {profile_row})', fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, image_size])
ax.set_ylim([0, 1.1])
plt.tight_layout()
plt.show()

## Convergence Analysis

### Expected Convergence Behavior

The Richardson-Lucy algorithm with TV regularization exhibits characteristic convergence behavior:

1. **Residual Decay**: The data fidelity term (residual) should decrease monotonically in early iterations, then plateau as the algorithm approaches the regularized solution

2. **TV Norm Evolution**: The TV norm typically increases initially as the algorithm sharpens edges, then stabilizes as regularization balances sharpening with smoothing

3. **Relative Change**: The relative change between successive iterates should decrease, indicating convergence to a fixed point

### Diagnosing Problems

- **Oscillating residual**: May indicate numerical instability or inappropriate regularization
- **Slow convergence**: Consider increasing the regularization parameter or using acceleration techniques
- **Divergence**: Check for division by zero, negative values, or PSF normalization issues

### Stopping Criteria

Common stopping criteria include:
- Fixed number of iterations
- Relative change below threshold: $\|x^{(n+1)} - x^{(n)}\| / \|x^{(n)}\| < \epsilon$
- Residual below threshold

In [ ]:
# ============================================
# Cell 11: Convergence Curve Plot
# ============================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Residual plot
ax = axes[0]
ax.semilogy(convergence_history['iteration'], convergence_history['residual'], 
            'b-o', linewidth=2, markersize=6)
ax.set_xlabel('Iteration')
ax.set_ylabel('Residual (log scale)')
ax.set_title('Data Fidelity (Residual)', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, max(convergence_history['iteration']) + 1])

# TV norm plot
ax = axes[1]
ax.plot(convergence_history['iteration'], convergence_history['tv_norm'], 
        'g-s', linewidth=2, markersize=6)
ax.set_xlabel('Iteration')
ax.set_ylabel('TV Norm')
ax.set_title('Total Variation Norm', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, max(convergence_history['iteration']) + 1])

# Relative change plot
ax = axes[2]
ax.semilogy(convergence_history['iteration'], convergence_history['change'], 
            'r-^', linewidth=2, markersize=6)
ax.set_xlabel('Iteration')
ax.set_ylabel('Relative Change (log scale)')
ax.set_title('Convergence (Relative Change)', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axhline(y=1e-3, color='k', linestyle='--', alpha=0.5, label='Typical threshold')
ax.legend()
ax.set_xlim([0, max(convergence_history['iteration']) + 1])

plt.suptitle('Convergence Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print convergence summary
print("\nConvergence Summary:")
print("="*50)
print(f"Initial residual:  {convergence_history['residual'][0]:.4e}")
print(f"Final residual:    {convergence_history['residual'][-1]:.4e}")
print(f"Residual reduction: {convergence_history['residual'][0]/convergence_history['residual'][-1]:.2f}x")
print(f"Final relative change: {convergence_history['change'][-1]:.6f}")
print("="*50)

## Error Analysis & Sensitivity

### Sources of Error

The reconstruction error arises from multiple sources:

1. **PSF Estimation Error**: Inaccurate PSF parameters lead to model mismatch
2. **Noise Amplification**: The inverse problem amplifies high-frequency noise
3. **Regularization Bias**: TV regularization introduces smoothing bias
4. **Discretization Error**: Grid-based PSF interpolation is an approximation
5. **Boundary Effects**: Padding strategies affect edge regions

### Noise Sensitivity

The condition number of the deconvolution problem depends on the PSF:
- Wider PSFs (more blur) → more ill-conditioned → higher noise sensitivity
- Narrower PSFs → better conditioned → more stable reconstruction

### Regularization Trade-off

The TV regularization parameter $\lambda$ controls the bias-variance trade-off:
- **Low $\lambda$**: Less bias, more noise, sharper but noisier result
- **High $\lambda$**: More bias, less noise, smoother but potentially over-smoothed

### Spatial Variation of Error

Error is expected to vary spatially:
- Regions with larger blur (bottom-right in our example) are harder to reconstruct
- Edge regions may show boundary artifacts
- High-contrast features may exhibit ringing

In [ ]:
# ============================================
# Cell 13: Error Map & Sensitivity Analysis
# ============================================

# Compute error maps
error_blurred = np.abs(gt_img - noisy_blurred_img)
error_reconstructed = np.abs(gt_img - reconstructed_img)

# Error statistics
print("Error Statistics:")
print("="*60)
print(f"Blurred Image Error:      Mean = {error_blurred.mean():.4f}, "
      f"Max = {error_blurred.max():.4f}, Std = {error_blurred.std():.4f}")
print(f"Reconstructed Error:      Mean = {error_reconstructed.mean():.4f}, "
      f"Max = {error_reconstructed.max():.4f}, Std = {error_reconstructed.std():.4f}")
print("="*60)

# Visualize error maps
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Error maps
ax = axes[0, 0]
im = ax.imshow(error_blurred, cmap='hot', vmin=0, vmax=0.3)
ax.set_title(f'Error: Blurred\nMean = {error_blurred.mean():.4f}', fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[0, 1]
im = ax.imshow(error_reconstructed, cmap='hot', vmin=0, vmax=0.3)
ax.set_title(f'Error: Reconstructed\nMean = {error_reconstructed.mean():.4f}', fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[0, 2]
improvement_map = error_blurred - error_reconstructed
im = ax.imshow(improvement_map, cmap='RdBu', vmin=-0.2, vmax=0.2)
ax.set_title('Error Improvement\n(Blue = Better)', fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Sensitivity study: vary regularization parameter
print("\nRunning sensitivity study (varying λ)...")
lambda_values = [0.01, 0.05, 0.1, 0.2]
psnr_values = []
ssim_values = []

for lbd in lambda_values:
    result, _ = rl_deconv_spatially_variant(
        image_masked_list, estimated_psf_list, 
        iterations=15, lbd=lbd, track_convergence=False
    )
    result = np.clip(result, 0, 1)
    psnr_val = metrics.peak_signal_noise_ratio(gt_img, result, data_range=1.0)
    ssim_val = metrics.structural_similarity(gt_img, result, data_range=1.0)
    psnr_values.append(psnr_val)
    ssim_values.append(ssim_val)
    print(f"  λ = {lbd:.2f}: PSNR = {psnr_val:.2f} dB, SSIM = {ssim_val:.4f}")

# Plot sensitivity
ax = axes[1, 0]
ax.plot(lambda_values, psnr_values, 'bo-', linewidth=2, markersize=8)
ax.set_xlabel('Regularization Parameter (λ)')
ax.set_ylabel('PSNR (dB)')
ax.set_title('PSNR vs Regularization', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axhline(y=psnr_blurred, color='r', linestyle='--', alpha=0.5, label='Blurred baseline')
ax.legend()

ax = axes[1, 1]
ax.plot(lambda_values, ssim_values, 'gs-', linewidth=2, markersize=8)
ax.set_xlabel('Regularization Parameter (λ)')
ax.set_ylabel('SSIM')
ax.set_title('SSIM vs Regularization', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axhline(y=ssim_blurred, color='r', linestyle='--', alpha=0.5, label='Blurred baseline')
ax.legend()

# Noise sensitivity study
print("\nRunning noise sensitivity study...")
noise_levels = [0.005, 0.01, 0.02, 0.05]
psnr_noise = []

for sigma in noise_levels:
    # Generate noisy observation
    noisy_obs = blurred_img + np.random.normal(0, sigma, blurred_img.shape)
    noisy_obs = np.clip(noisy_obs, 0, 1)
    
    # Prepare weighted images
    masked_list = [np.multiply(grid_weights[i], noisy_obs) 
                   for i in range(len(estimated_psf_list))]
    
    result, _ = rl_deconv_spatially_variant(
        masked_list, estimated_psf_list, 
        iterations=15, lbd=0.05, track_convergence=False
    )
    result = np.clip(result, 0, 1)
    psnr_val = metrics.peak_signal_noise_ratio(gt_img, result, data_range=1.0)
    psnr_noise.append(psnr_val)
    print(f"  σ_noise = {sigma:.3f}: PSNR = {psnr_val:.2f} dB")

ax = axes[1, 2]
ax.semilogx(noise_levels, psnr_noise, 'r^-', linewidth=2, markersize=8)
ax.set_xlabel('Noise Level (σ)')
ax.set_ylabel('PSNR (dB)')
ax.set_title('Noise Sensitivity', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.invert_xaxis()  # Higher noise on left

plt.suptitle('Error Analysis and Sensitivity Study', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Discussion & Key Takeaways

### Summary of Findings

This tutorial demonstrated semi-blind PSF deconvolution for microscopy images with spatially variant blur. Key observations:

1. **Effective Deblurring**: The Richardson-Lucy algorithm with TV regularization successfully recovers image details lost to spatially variant blur

2. **PSF Estimation Importance**: Even with estimation errors in PSF parameters, reasonable reconstruction is achievable, demonstrating robustness of the approach

3. **Regularization Trade-off**: The TV parameter $\lambda$ critically affects the bias-variance trade-off; optimal values depend on noise level and image content

### Limitations

1. **Computational Cost**: Spatially variant deconvolution requires multiple convolutions per iteration, scaling with grid resolution

2. **PSF Model Assumptions**: Gaussian PSF assumption may not hold for all optical systems (e.g., aberrated systems with complex PSF shapes)

3. **Grid Resolution**: Coarse PSF grids may miss rapid spatial variations; fine grids increase computation

4. **Noise Model**: Gaussian noise assumption; Poisson noise (photon counting) may require modified algorithms

### Extensions and Improvements

1. **Deep Learning Integration**: Use CNNs for both PSF estimation and image reconstruction (end-to-end learning)

2. **Adaptive Regularization**: Spatially varying regularization based on local image content

3. **3D Extension**: Extend to volumetric microscopy with depth-dependent PSF

4. **Acceleration**: GPU implementation, conjugate gradient methods, or learned optimization

### Key References

1. Richardson, W.H. (1972). "Bayesian-based iterative method of image restoration." *Journal of the Optical Society of America*, 62(1):55-59.

2. Dey, N. et al. (2006). "Richardson-Lucy algorithm with total variation regularization for 3D confocal microscope deconvolution." *Microscopy Research and Technique*, 69(4):260-266.

3. Lauer, T. (2002). "Deconvolution with a spatially-variant PSF." *SPIE Astronomical Telescopes + Instrumentation*, 4847:167-173.

In [ ]:
# ============================================
# Cell 15: Summary Metrics Table
# ============================================

print("\n" + "="*70)
print("           SEMI-BLIND PSF DECONVOLUTION - RESULTS SUMMARY")
print("="*70)

print("\n" + "-"*70)
print("                        IMAGE QUALITY METRICS")
print("-"*70)
print(f"{'Metric':<25} {'Blurred':<15} {'Reconstructed':<15} {'Improvement':<15}")
print("-"*70)
print(f"{'PSNR (dB)':<25} {psnr_blurred:<15.2f} {psnr_reconstructed:<15.2f} "
      f"{psnr_reconstructed - psnr_blurred:+.2f}")
print(f"{'SSIM':<25} {ssim_blurred:<15.4f} {ssim_reconstructed:<15.4f} "
      f"{ssim_reconstructed - ssim_blurred:+.4f}")
print(f"{'Mean Abs Error':<25} {error_blurred.mean():<15.4f} {error_reconstructed.mean():<15.4f} "
      f"{error_reconstructed.mean() - error_blurred.mean():+.4f}")
print(f"{'Max Abs Error':<25} {error_blurred.max():<15.4f} {error_reconstructed.max():<15.4f} "
      f"{error_reconstructed.max() - error_blurred.max():+.4f}")

print("\n" + "-"*70)
print("                        ALGORITHM PARAMETERS")
print("-"*70)
print(f"{'Parameter':<35} {'Value':<35}")
print("-"*70)
print(f"{'Image Size':<35} {f'{image_size} x {image_size} pixels':<35}")
print(f"{'PSF Grid':<35} {f'{grid_rows} x {grid_cols}':<35}")
print(f"{'PSF Kernel Size':<35} {f'{psf_size} x {psf_size} pixels':<35}")
print(f"{'RL Iterations':<35} {'20':<35}")
print(f"{'TV Regularization (λ)':<35} {'0.05':<35}")
print(f"{'Noise Level (σ)':<35} {f'{noise_sigma}':<35}")

print("\n" + "-"*70)
print("                        CONVERGENCE STATISTICS")
print("-"*70)
print(f"{'Metric':<35} {'Value':<35}")
print("-"*70)
print(f"{'Initial Residual':<35} {convergence_history['residual'][0]:.4e}")
print(f"{'Final Residual':<35} {convergence_history['residual'][-1]:.4e}")
print(f"{'Residual Reduction Factor':<35} "
      f"{convergence_history['residual'][0]/convergence_history['residual'][-1]:.2f}x")
print(f"{'Final Relative Change':<35} {convergence_history['change'][-1]:.6f}")

print("\n" + "-"*70)
print("                    SENSITIVITY ANALYSIS RESULTS")
print("-"*70)
print(f"{'λ Value':<20} {'PSNR (dB)':<20} {'SSIM':<20}")
print("-"*70)
for lbd, psnr, ssim in zip(lambda_values, psnr_values, ssim_values):
    print(f"{lbd:<20.2f} {psnr:<20.2f} {ssim:<20.4f}")

print("\n" + "="*70)
print("                           END OF SUMMARY")
print("="*70 + "\n")

## Conclusion

This tutorial presented a comprehensive implementation of **semi-blind PSF deconvolution** for microscopy images with spatially variant blur.

### Problem Addressed

We tackled the inverse problem of recovering a sharp image from observations degraded by:
- Spatially varying point spread functions (PSFs)
- Additive Gaussian noise

### Method Used

Our approach combined:
1. **Grid-based PSF interpolation** to model spatial variation
2. **Richardson-Lucy deconvolution** for iterative image reconstruction
3. **Total Variation regularization** for noise suppression
4. **Deep learning-based PSF estimation** (simulated in this tutorial)

### Key Results

The reconstruction achieved significant improvement over the blurred observation:
- PSNR improvement of several dB
- SSIM improvement demonstrating better structural fidelity
- Robust performance even with PSF estimation errors

### Practical Implications

This technique is applicable to:
- Fluorescence microscopy with field-dependent aberrations
- Adaptive optics systems with residual aberrations
- Any imaging system with known or estimable spatially variant blur

The alternating optimization framework provides a flexible approach that can incorporate various PSF estimation methods and regularization strategies, making it adaptable to diverse imaging scenarios.